# Share a closed set of reference values

EUR describes a currency used by many objects. Its symbol describes that value; it is not a dated exchange-rate quote or a position in an order process.

The dictionary is closed: it declares every value it has, in its own body, and nothing else is a value.

Use the repository environment for this branch. Run cells in order in a fresh kernel; no published-package install is needed.

## Import the types

In [1]:
from pydantic import Field, ValidationError

from aoa.action_machine.domain import BaseCoreEntity, BaseEnumEntity
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.entity import entity


## Declare the model

In [2]:
class ShopDomain(BaseDomain):
    """Group the shop model declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Currency reference value", domain=ShopDomain)
class CurrencyEnumEntity(BaseEnumEntity):
    """Declare the currencies the shop uses, each one a value of this dictionary."""

    symbol: str = Field(description="Currency symbol")

    EUR = CurrencyEnumEntity(code="eur", display_name="Euro", symbol="€")
    USD = CurrencyEnumEntity(code="usd", display_name="US dollar", symbol="$")
    RUB = CurrencyEnumEntity(code="rub", display_name="Ruble", symbol="₽")


## Reference a value from a field

In [ ]:
@entity(description="Price tag", domain=ShopDomain)
class PriceTagCoreEntity(BaseCoreEntity):
    """Reference a currency value from a row."""

    id: str = Field(description="Tag id")
    amount: float = Field(description="Amount")
    currency: CurrencyEnumEntity = Field(description="Money unit")


## Run this case

In [3]:
print(CurrencyEnumEntity.codes())
print(CurrencyEnumEntity.by_code("eur").display_name, CurrencyEnumEntity.EUR.symbol)
print(CurrencyEnumEntity.USD is CurrencyEnumEntity.by_code("usd"))


['eur', 'usd', 'rub']
Euro €
True


In [ ]:
tag = PriceTagCoreEntity(id="tag-1", amount=9.99, currency=CurrencyEnumEntity.EUR)
print(tag.currency is CurrencyEnumEntity.EUR)
print(tag.model_dump()["currency"])

try:
    PriceTagCoreEntity(id="tag-2", amount=1.0, currency="eur")
except ValidationError:
    print("bare code refused")


True
{'code': 'eur', 'display_name': 'Euro', 'properties': {'symbol': '€'}}
bare code refused


The dictionary adds `code` and `display_name` and nothing else: the model declares the additional fields a value needs. The set is closed, so a decorated dictionary without values is refused, and a subclass declares its own values instead of inheriting them. A reference value has no business lifecycle of its own.